# CIFAR-10 Image Classification with a CNN: Project Part 2

This notebook builds a CNN (Convolutional Neural Network) that recognises the ten CIFAR-10 classes, measures how accurate it is, tests it on three of my own photos and on photos taken by the e-puck robot's camera in Webots, and measures ways to make the model fast enough to run on a robot.

**About the dataset**

CIFAR-10 contains 60,000 colour images of 32 × 32 pixels in 10 classes: 50,000 images for training and 10,000 for testing. The classes are:
* airplane
* automobile
* bird
* cat
* deer
* dog
* frog
* horse
* ship
* truck

**What this notebook does**
1. Setup
2. Load and inspect the data
3. Baseline CNN: a simple CNN to compare against
   * 3b. Optimiser comparison: SGD vs Adam
4. Improved CNN: all the layers of the model
5. Training the improved CNN
6. Accuracy and error analysis
7. Predictions on my own images
8. Predictions on images from the robot's camera (Webots)
9. Optimisation for running on the robot
10. Summary of results
11. Save the outputs

**How to run**
1. In Colab: `Runtime → Change runtime type → T4 GPU → Save`.
2. `Runtime → Run all`. When asked, allow access to Google Drive: the trained models are saved there.
   * **First run:** the models are trained (about 15–20 minutes on a T4 GPU) and saved to `MyDrive/robot_vision_cnn/`.
   * **Every later run:** the models are loaded from Drive in a few seconds and training is skipped, so the whole notebook takes about 5 minutes. To train again from scratch, set `TRAIN_AGAIN = True` in the settings cell.
3. Section 7: when the **Choose files** button appears, select three of my own photos, each showing one object from the 10 classes, roughly in the centre.
4. Section 8: select the four photos saved by the Webots controller, `NNN_airplane_photo.png`, `NNN_bird_photo.png`, `NNN_truck_photo.png` and `NNN_horse_photo.png`, from `controllers/robot_vision_controller/images/`. They are already cropped around the object.
5. At the end, `assignment3_outputs.zip` is downloaded with all the figures, the trained models and `results_summary.json`.

## 1. Setup
Imports the libraries, sets the main settings and connects Google Drive. Figures are saved in `figures/` and trained models in `models/`. Because Colab deletes its files when the session ends, the trained models are also kept in Google Drive, so they do not have to be trained again in every session.

In [ ]:
import os, io, json, time, zipfile, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import confusion_matrix, classification_report
try:
    from IPython.display import display
except ImportError:
    display = print
tf.get_logger().setLevel("ERROR")          # hide TensorFlow's information messages

SEED = 42
keras.utils.set_random_seed(SEED)          # same random numbers on every run, so the results can be repeated

# ------------------------------ Settings ------------------------------
EPOCHS        = 45      # maximum passes over the training data; early stopping may end training sooner
BATCH_SIZE    = 128     # images per training step
VAL_SPLIT     = 5000    # training images kept aside to check the model during training (validation set)
QUICK_TEST    = False   # True = 2 epochs on a small subset, only to check that every cell works
TRAIN_AGAIN   = False   # True = train both models again even if trained models are saved in Google Drive
FIG_DIR       = pathlib.Path("figures");   FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR     = pathlib.Path("models");    MODEL_DIR.mkdir(exist_ok=True)
OWN_IMG_DIR   = pathlib.Path("my_images"); OWN_IMG_DIR.mkdir(exist_ok=True)
WEBOTS_IMG_DIR= pathlib.Path("webots_images"); WEBOTS_IMG_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",     # in the order of the labels 0-9
               "dog", "frog", "horse", "ship", "truck"]

# Figure style: sharp images for the slides, no frame lines on the top and right
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False})
RESULTS = {}   # the key numbers are collected here and saved to results_summary.json at the end

# ------------------------------ Google Drive ------------------------------
# Colab deletes its files when the session ends, so the trained models and their training histories are
# kept in Google Drive. Quick-test models go to a separate folder so they never replace the real ones.
DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = pathlib.Path("/content/drive/MyDrive") / ("robot_vision_cnn_quicktest" if QUICK_TEST else "robot_vision_cnn")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Trained models are kept in", DRIVE_DIR)
except Exception as e:                          # not in Colab, or access to Drive not allowed
    print("Google Drive not available - the models will be trained in this session only:", e)


def load_from_drive(model_file, history_file):
    """Load a trained model and its training history from Google Drive.
    Returns (model, history, training minutes), or None if there is nothing to load or TRAIN_AGAIN is True."""
    if DRIVE_DIR is None or TRAIN_AGAIN or not (DRIVE_DIR / model_file).exists() or not (DRIVE_DIR / history_file).exists():
        return None
    info = json.loads((DRIVE_DIR / history_file).read_text())
    print(f"Loaded {model_file} from Google Drive - training skipped")
    return keras.models.load_model(DRIVE_DIR / model_file), info["history"], info["train_time_min"]


def save_to_drive(trained_model, model_file, history, history_file, minutes):
    """Keep a trained model and its training history in Google Drive for later sessions."""
    if DRIVE_DIR is None:
        return
    trained_model.save(DRIVE_DIR / model_file)
    (DRIVE_DIR / history_file).write_text(json.dumps({"history": history, "train_time_min": minutes}, default=float))
    print(f"Saved {model_file} to Google Drive")

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none - switch the runtime to a GPU")

## 2. Load and inspect the data
CIFAR-10 (170 MB) is loaded from the fastest place available:
1. the copy saved in Google Drive by an earlier session (a few seconds);
2. otherwise the Hugging Face copy of CIFAR-10 (a fast download server);
3. if that fails, `keras.datasets`, which downloads from the original University of Toronto server (sometimes slow).

After the first download, a copy is saved in Google Drive, so later sessions do not download it again.

5,000 of the 50,000 training images are kept aside as a **validation set**. Every decision during training (when to lower the learning rate, when to stop) uses the validation set only, so the 10,000 test images are used just once, at the end, to give an honest accuracy score.

In [ ]:
def load_cifar10():
    """CIFAR-10 as arrays: images (N, 32, 32, 3) with pixels 0-255 and labels 0-9. Uses the copy in Google Drive
    if there is one, otherwise downloads it (Hugging Face copy first, original server if that fails) and keeps
    a copy in Google Drive for the next sessions."""
    cache = DRIVE_DIR / "cifar10.npz" if DRIVE_DIR else None
    if cache and cache.exists():
        d = np.load(cache)
        print("CIFAR-10 loaded from Google Drive")
        return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
    try:
        data = []
        for split in ("train", "test"):              # one file per split; each image is stored as PNG bytes
            url = f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet"
            df = pd.read_parquet(keras.utils.get_file(f"cifar10_{split}.parquet", url))
            x = np.stack([np.array(Image.open(io.BytesIO(img["bytes"])).convert("RGB")) for img in df["img"]])
            data.append((x, df["label"].to_numpy().astype("uint8")))
        assert data[0][0].shape == (50000, 32, 32, 3) and data[1][0].shape == (10000, 32, 32, 3)
        print("CIFAR-10 downloaded from the Hugging Face copy")
    except Exception as e:                           # copy not reachable or a different format
        print(f"Hugging Face copy not available ({type(e).__name__}) - downloading from the original server")
        (xa, ya), (xb, yb) = keras.datasets.cifar10.load_data()
        data = [(xa, ya.flatten()), (xb, yb.flatten())]
    if cache:
        np.savez_compressed(cache, x_train=data[0][0], y_train=data[0][1], x_test=data[1][0], y_test=data[1][1])
        print("Saved a copy of CIFAR-10 in Google Drive for the next sessions")
    return data[0], data[1]


(x_train_full, y_train_full), (x_test, y_test) = load_cifar10()                    # pixels 0-255, labels 0-9
y_train_full, y_test = y_train_full.flatten(), y_test.flatten()                       # labels as a flat list

x_train, y_train = x_train_full[:-VAL_SPLIT], y_train_full[:-VAL_SPLIT]   # first 45,000 images: training
x_val,   y_val   = x_train_full[-VAL_SPLIT:], y_train_full[-VAL_SPLIT:]   # last 5,000: validation

if QUICK_TEST:                     # small subset, only to check that the cells run
    x_train, y_train, x_val, y_val = x_train[:2000], y_train[:2000], x_val[:500], y_val[:500]
    x_test, y_test = x_test[:1000], y_test[:1000]
    EPOCHS = 2

print(f"train {x_train.shape}  val {x_val.shape}  test {x_test.shape}  dtype {x_train.dtype}")

# Sample grid: 8 training images of each class, one class per row
fig, axes = plt.subplots(10, 8, figsize=(8, 10.5))
for c in range(10):
    idx = np.where(y_train == c)[0][:8]
    for j, i in enumerate(idx):
        ax = axes[c, j]; ax.imshow(x_train[i]); ax.axis("off")
    axes[c, 0].text(-4, 16, CLASS_NAMES[c], ha="right", va="center", fontsize=9)
plt.suptitle("CIFAR-10 samples (32×32 px)", y=0.995)
plt.savefig(FIG_DIR / "fig01_dataset_samples.png"); plt.show()

## 3. Baseline CNN
First, a simple CNN is trained to give a result to improve on. It is the small network from the TensorFlow CNN tutorial:

`Conv2D(32) → MaxPooling2D → Conv2D(64) → MaxPooling2D → Conv2D(64) → Flatten → Dense(64) → Dense(10)`

* The `Rescaling(1/255)` layer turns pixel values 0–255 into 0–1 inside the model, so the model can be given raw camera pixels.
* The last layer has no softmax: it outputs raw scores (logits), so the loss is set with `from_logits=True`.
* It is trained with the Adam optimiser for 10 epochs and checked on the validation set, not the test set, so its test accuracy can be compared fairly with the improved model.

In [ ]:
BASELINE_EPOCHS = 2 if QUICK_TEST else 10
baseline = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Rescaling(1.0 / 255),                      # pixels 0-255 -> 0-1
    layers.Conv2D(32, (3, 3), activation="relu"),     # 32 filters that respond to simple patterns (edges, colours)
    layers.MaxPooling2D((2, 2)),                      # halve the width and height, keep the strongest responses
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.Flatten(),                                 # all feature maps -> one long vector
    layers.Dense(64, activation="relu"),
    layers.Dense(10),                                 # one score (logit) per class
], name="baseline_cnn")
baseline.summary()                                    # every layer with its output shape and parameters

saved = load_from_drive("baseline_cnn.keras", "baseline_history.json")
if saved:                                             # trained in an earlier session: no training needed
    baseline, bh, b_minutes = saved
else:                                                 # first run (or TRAIN_AGAIN): train and keep it in Drive
    baseline.compile(optimizer="adam",
                     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    t0 = time.time()
    bh = baseline.fit(x_train.astype("float32"), y_train, epochs=BASELINE_EPOCHS,
                      validation_data=(x_val.astype("float32"), y_val), verbose=2).history
    b_minutes = round((time.time() - t0) / 60, 1)
    save_to_drive(baseline, "baseline_cnn.keras", bh, "baseline_history.json", b_minutes)
b_loss, b_acc = baseline.evaluate(x_test.astype("float32"), y_test, verbose=0)   # score on the 10,000 test images
baseline.save(MODEL_DIR / "baseline_cnn.keras")
RESULTS.update(baseline_test_accuracy=round(b_acc * 100, 2), baseline_params=int(baseline.count_params()),
               baseline_train_time_min=b_minutes,
               baseline_final_train_acc=round(bh["accuracy"][-1] * 100, 2),
               baseline_final_val_acc=round(bh["val_accuracy"][-1] * 100, 2))
print(f"Baseline CNN test accuracy: {b_acc*100:.2f}%  ({baseline.count_params():,} params)")

# Training curves: training accuracy rising while validation accuracy levels off means overfitting
ep = range(1, len(bh["accuracy"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, bh["accuracy"], label="train"); a1.plot(ep, bh["val_accuracy"], label="validation")
a1.set_title("Accuracy"); a1.set_xlabel("epoch"); a1.legend(); a1.grid(alpha=.3)
a2.plot(ep, bh["loss"], label="train"); a2.plot(ep, bh["val_loss"], label="validation")
a2.set_title("Loss"); a2.set_xlabel("epoch"); a2.legend(); a2.grid(alpha=.3)
plt.suptitle("Baseline CNN: the gap between training and validation shows overfitting")
plt.savefig(FIG_DIR / "fig02b_baseline_curves.png"); plt.show()

## 3b. Optimiser comparison: SGD vs Adam
The optimiser decides how the weights are changed after each batch:
* **SGD** (stochastic gradient descent) moves every weight a small, fixed step (learning rate 0.01) against its gradient;
* **Adam** gives every weight its own step size, based on the average and the spread of its recent gradients (learning rate 0.001).

To compare them fairly, the baseline CNN is trained twice from **exactly the same starting weights**, once with each optimiser, for 10 epochs, with the same data and validation set. Only the optimiser changes.

In [ ]:
OPT_EPOCHS = 2 if QUICK_TEST else 10
OPTIMISERS = {"SGD": lambda: keras.optimizers.SGD(learning_rate=0.01),     # the Keras defaults for "sgd"
              "Adam": lambda: keras.optimizers.Adam(learning_rate=0.001)}  # and "adam"

opt_hist, opt_models, opt_minutes = {}, {}, {}
saved = {n: load_from_drive(f"opt_{n.lower()}.keras", f"opt_{n.lower()}_history.json") for n in OPTIMISERS}
if all(saved.values()):                          # trained in an earlier session: no training needed
    for n, (m, h, minutes) in saved.items():
        opt_models[n], opt_hist[n], opt_minutes[n] = m, h, minutes
else:
    start_weights = keras.models.clone_model(baseline).get_weights()   # one random start, used by both
    for n, make_optimiser in OPTIMISERS.items():
        m = keras.models.clone_model(baseline)                         # same layers as the baseline CNN
        m.set_weights(start_weights)
        m.compile(optimizer=make_optimiser(),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
        t0 = time.time()
        opt_hist[n] = m.fit(x_train.astype("float32"), y_train, epochs=OPT_EPOCHS,
                            validation_data=(x_val.astype("float32"), y_val), verbose=2).history
        opt_minutes[n] = round((time.time() - t0) / 60, 1)
        opt_models[n] = m
        save_to_drive(m, f"opt_{n.lower()}.keras", opt_hist[n], f"opt_{n.lower()}_history.json", opt_minutes[n])

# Results table: final and best validation accuracy, and the score on the 10,000 test images
rows = []
for n, h in opt_hist.items():
    best = int(np.argmax(h["val_accuracy"]))
    rows.append({"Optimiser": n,
                 "Final train acc (%)": round(h["accuracy"][-1] * 100, 2),
                 "Final val acc (%)": round(h["val_accuracy"][-1] * 100, 2),
                 "Best val acc (%)": round(h["val_accuracy"][best] * 100, 2), "Best epoch": best + 1,
                 "Test acc (%)": round(opt_models[n].evaluate(x_test.astype("float32"), y_test, verbose=0)[1] * 100, 2),
                 "Train time (min)": opt_minutes[n]})
RESULTS["optimiser_comparison"] = rows
display(pd.DataFrame(rows))

# One graph for both optimisers: solid = validation, dashed = training
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
colours = {"SGD": "#e8590c", "Adam": "#1971c2"}
for n, h in opt_hist.items():
    ep = range(1, len(h["accuracy"]) + 1)
    a1.plot(ep, h["accuracy"], "--", color=colours[n], label=f"{n} train")
    a1.plot(ep, h["val_accuracy"], "-o", ms=3, color=colours[n], label=f"{n} validation")
    a2.plot(ep, h["loss"], "--", color=colours[n], label=f"{n} train")
    a2.plot(ep, h["val_loss"], "-o", ms=3, color=colours[n], label=f"{n} validation")
a1.set_title("Accuracy"); a2.set_title("Loss")
for a in (a1, a2):
    a.set_xlabel("epoch"); a.legend(fontsize=8); a.grid(alpha=.3)
plt.suptitle("SGD vs Adam on the baseline CNN (same starting weights, same data)")
plt.savefig(FIG_DIR / "fig02c_sgd_vs_adam.png"); plt.show()

**What to look for.** Adam usually climbs much faster in the first epochs and ends with a higher accuracy, because each weight gets a step size that suits it, while SGD with one fixed step size learns slowly and steadily. Watch also where the validation curve of Adam stops rising while its training curve keeps going up: that gap is the start of overfitting, which the improved CNN in section 4 is designed to reduce. Adam is therefore used for the improved CNN.

## 4. Improved CNN: model layers
The baseline's training curves show it starting to overfit after a few epochs: training accuracy keeps rising while validation accuracy levels off. The improved model is designed to reduce this, while staying small enough to run on a robot (about 0.3 million parameters).

* **3 convolutional blocks** with 32 → 64 → 128 filters. Each block is `Conv → BatchNorm → ReLU`, twice, then `MaxPooling` and `Dropout`. Two convolutions per block learn richer features before the image is made smaller; `padding="same"` keeps the size inside a block.
* **Batch normalisation** keeps the values passed between layers in a stable range, so training is faster and more stable.
* **Dropout** (0.2, 0.3, 0.4 in the blocks and 0.5 before the output) randomly switches off units during training, so the model cannot rely on any single feature. This reduces overfitting.
* **L2 weight decay** (1e-4) on the convolution weights also discourages overfitting.
* **Global average pooling** replaces `Flatten`: each of the 128 feature maps is averaged to one number. This shrinks the classifier part from about 262,000 to about 16,500 parameters.
* **Softmax output** gives probabilities, which can be read as the model's confidence.

The next cells print every layer with its output shape and number of parameters.

In [ ]:
def conv_block(x, filters, dropout, name):
    """Two 3x3 convolutions, each followed by batch normalisation and ReLU, then max pooling and dropout."""
    for k in (1, 2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False,      # "same" keeps the image size;
                          kernel_regularizer=keras.regularizers.l2(1e-4),  # no bias needed before batch norm
                          name=f"{name}_conv{k}")(x)                        # L2 weight decay against overfitting
        x = layers.BatchNormalization(name=f"{name}_bn{k}")(x)             # keeps values in a stable range
        x = layers.Activation("relu", name=f"{name}_relu{k}")(x)
    x = layers.MaxPooling2D(2, name=f"{name}_pool")(x)                     # halve the width and height
    return layers.Dropout(dropout, name=f"{name}_drop")(x)                 # switch off random units while training


def build_model():
    """Improved CNN: 3 convolution blocks (32, 64, 128 filters), global average pooling, one dense layer
    and a softmax output that gives a probability for each class."""
    inputs = keras.Input(shape=(32, 32, 3), name="image_0_255")            # raw pixels 0-255
    x = layers.Rescaling(1.0 / 255, name="rescale")(inputs)                 # -> 0-1
    x = conv_block(x, 32,  0.2, "block1")
    x = conv_block(x, 64,  0.3, "block2")
    x = conv_block(x, 128, 0.4, "block3")
    x = layers.GlobalAveragePooling2D(name="gap")(x)                        # each feature map -> one number
    x = layers.Dense(128, activation="relu", name="fc")(x)
    x = layers.Dropout(0.5, name="fc_drop")(x)
    outputs = layers.Dense(10, activation="softmax", name="class_probs")(x)
    return keras.Model(inputs, outputs, name="cifar10_vgg_bn")


model = build_model()
model.summary(show_trainable=True)        # every layer with its output shape and number of parameters

In [ ]:
# Table of every layer (name, type, output shape, parameters), shown here and saved as a figure for the slides
rows = []
for l in model.layers:
    try:
        out = tuple(l.output.shape)
    except Exception:
        out = "-"
    rows.append({"#": len(rows), "Layer": l.name, "Type": l.__class__.__name__,
                 "Output shape": str(out).replace("None", "N"), "Params": l.count_params()})
layer_df = pd.DataFrame(rows)
display(layer_df)

total_params = model.count_params()
RESULTS["total_params"] = int(total_params)
RESULTS["num_layers"] = len(model.layers)

fig, ax = plt.subplots(figsize=(9, 0.32 * len(layer_df) + 0.8)); ax.axis("off")
cells_txt = [[r["#"], r["Layer"], r["Type"], r["Output shape"], f'{r["Params"]:,}'] for _, r in layer_df.iterrows()]
tbl = ax.table(cellText=cells_txt, colLabels=layer_df.columns, cellLoc="left",
               colWidths=[0.06, 0.2, 0.32, 0.24, 0.18], bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
for (r, c), cell in tbl.get_celld().items():
    if r == 0: cell.set_facecolor("#2b4c7e"); cell.set_text_props(color="white", weight="bold")
ax.set_title(f"All {len(layer_df)} layers — {total_params:,} parameters", pad=8)
plt.savefig(FIG_DIR / "fig02_layer_table.png"); plt.show()

# Architecture diagram (needs pydot and graphviz, both already installed on Colab)
try:
    keras.utils.plot_model(model, to_file=str(FIG_DIR / "fig02_architecture.png"),
                           show_shapes=True, show_layer_names=True, dpi=110)
    print("Saved architecture diagram -> figures/fig02_architecture.png")
except Exception as e:
    print("plot_model skipped:", e)

## 5. Training the improved CNN
* **Data augmentation** (training images only, new variations every epoch): random horizontal flip, random shift of up to 4 pixels, and small brightness and contrast changes. This helps the model generalise, and the lighting changes also prepare it for the different lighting in Webots.
* **Optimiser:** Adam with learning rate 0.001. `ReduceLROnPlateau` halves the learning rate when the validation loss has not improved for 4 epochs.
* **Early stopping:** training stops when validation accuracy has not improved for 10 epochs, and the best weights are restored.
* At most 45 epochs, batch size 128, loss: sparse categorical cross-entropy. With augmentation and dropout the model learns more slowly than the baseline, so it needs more epochs; 45 leaves room for the learning-rate reductions to take effect, and early stopping ends training earlier if validation accuracy stops improving.

If a trained model is already saved in Google Drive, it is loaded instead and the training curves are drawn from its saved history.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE                  # let TensorFlow decide how many images to prepare in parallel


def augment(img, label):
    """Random changes to a training image, so every epoch sees slightly different versions of it."""
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)               # mirror the image half of the time
    img = tf.image.pad_to_bounding_box(img, 4, 4, 40, 40)    # add a 4-pixel border, then cut a random
    img = tf.image.random_crop(img, (32, 32, 3))             # 32x32 piece: a shift of up to 4 pixels
    img = tf.image.random_brightness(img, 20.0)              # brighter or darker by up to 20 (of 255)
    img = tf.image.random_contrast(img, 0.85, 1.15)          # contrast changed by up to 15 %
    return tf.clip_by_value(img, 0.0, 255.0), label          # keep the pixel values in 0-255


def to_float(img, label):
    """Validation and test images are not changed, only converted to float."""
    return tf.cast(img, tf.float32), label


# Input pipelines: the training images are shuffled and augmented; validation and test images are used as they are
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
            .shuffle(10_000, seed=SEED).map(augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds  = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(to_float).batch(256).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(to_float).batch(256).prefetch(AUTOTUNE)

model.compile(optimizer=keras.optimizers.Adam(1e-3),            # Adam, learning rate 0.001
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # loss for labels 0-9

callbacks = [
    # halve the learning rate when the validation loss has not improved for 4 epochs
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    # stop after 10 epochs without a better validation accuracy and go back to the best weights
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
    # keep a copy of the best model so far
    keras.callbacks.ModelCheckpoint(str(MODEL_DIR / "best.keras"), monitor="val_accuracy", save_best_only=True),
]

# The saved files are named after the epoch limit, so changing EPOCHS trains a new model instead of loading an old one
saved = load_from_drive(f"cifar10_cnn_e{EPOCHS}.keras", f"cifar10_history_e{EPOCHS}.json")
if saved:                                     # trained in an earlier session: no training needed
    model, h, RESULTS["train_time_min"] = saved
else:                                         # first run (or TRAIN_AGAIN): train and keep it in Drive
    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2).history
    RESULTS["train_time_min"] = round((time.time() - t0) / 60, 1)
    save_to_drive(model, f"cifar10_cnn_e{EPOCHS}.keras", h, f"cifar10_history_e{EPOCHS}.json", RESULTS["train_time_min"])
RESULTS["epochs_run"] = len(h["loss"])        # h = training history, also used in the accuracy section
print(f"Training took {RESULTS['train_time_min']} min over {RESULTS['epochs_run']} epochs")

In [ ]:
# Training curves of the improved model
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(h["accuracy"], label="train"); a1.plot(h["val_accuracy"], label="validation")
a1.set_title("Accuracy"); a1.set_xlabel("epoch"); a1.legend(); a1.grid(alpha=.3)
a2.plot(h["loss"], label="train"); a2.plot(h["val_loss"], label="validation")
a2.set_title("Loss"); a2.set_xlabel("epoch"); a2.legend(); a2.grid(alpha=.3)
plt.suptitle("Training curves"); plt.savefig(FIG_DIR / "fig03_training_curves.png"); plt.show()

## 6. Accuracy and error analysis
The final accuracy is measured once, on the 10,000 test images. Besides the overall score, we look at the accuracy for each class, the confusion matrix (which classes are mixed up), the most confident mistakes, and a comparison with the baseline and with published results.

In [ ]:
# Final evaluation on the 10,000 test images (used only here)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
probs  = model.predict(test_ds, verbose=0)          # probability of each class for every test image
y_pred = probs.argmax(1)                            # the most likely class

RESULTS["test_accuracy"]  = round(float(test_acc) * 100, 2)
RESULTS["test_loss"]      = round(float(test_loss), 4)
RESULTS["best_val_accuracy"] = round(float(max(h["val_accuracy"])) * 100, 2)
RESULTS["train_accuracy_final"] = round(float(h["accuracy"][-1]) * 100, 2)

print("=" * 46)
print(f"   TEST ACCURACY : {test_acc*100:6.2f} %   (10,000 unseen images)")
print(f"   TEST LOSS     : {test_loss:.4f}")
print("=" * 46)
# Precision, recall and F1 score for each class
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3, output_dict=True)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Confusion matrix: rows = true class, columns = predicted class; each row is scaled to add up to 1
cm = confusion_matrix(y_test, y_pred, labels=range(10))
cm_norm = cm / cm.sum(1, keepdims=True)
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, cbar_kws={"label": "fraction of true class"})
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix — test accuracy {test_acc*100:.1f}%")
plt.savefig(FIG_DIR / "fig04_confusion_matrix.png"); plt.show()

# The five class pairs mixed up most often
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], CLASS_NAMES[i], CLASS_NAMES[j]) for i in range(10) for j in range(10)), reverse=True)[:5]
RESULTS["top_confusions"] = [f"{t} -> {p} ({n})" for n, t, p in pairs]
print("Top confusions (true -> predicted, count):"); [print("  ", s) for s in RESULTS["top_confusions"]]

# Accuracy of each class, lowest first
per_class = cm.diagonal() / cm.sum(1)
RESULTS["per_class_accuracy"] = {c: round(float(a) * 100, 1) for c, a in zip(CLASS_NAMES, per_class)}
order = np.argsort(per_class)
plt.figure(figsize=(8, 4))
bars = plt.barh(np.array(CLASS_NAMES)[order], per_class[order] * 100, color="#4c78a8")
plt.axvline(test_acc * 100, color="#e45756", ls="--", label=f"overall {test_acc*100:.1f}%")
for b in bars: plt.text(b.get_width() + 0.5, b.get_y() + b.get_height()/2, f"{b.get_width():.1f}", va="center", fontsize=8)
plt.xlim(0, 105); plt.xlabel("accuracy (%)"); plt.title("Per-class test accuracy"); plt.legend(loc="lower right")
plt.savefig(FIG_DIR / "fig05_per_class_accuracy.png"); plt.show()

In [ ]:
# The 16 mistakes the model was most sure about: they show what it finds hard
wrong = np.where(y_pred != y_test)[0]
conf_wrong = probs[wrong, y_pred[wrong]]
show = wrong[np.argsort(-conf_wrong)][:16]
fig, axes = plt.subplots(2, 8, figsize=(14, 4.2))
for ax, i in zip(axes.flat, show):
    ax.imshow(x_test[i]); ax.axis("off")
    ax.set_title(f"T: {CLASS_NAMES[y_test[i]]}\nP: {CLASS_NAMES[y_pred[i]]} {probs[i].max()*100:.0f}%", fontsize=8, color="#b22")
plt.suptitle("Most confident misclassifications (T = true, P = predicted)")
plt.savefig(FIG_DIR / "fig06_misclassified.png"); plt.show()

In [ ]:
# Accuracy compared with the baseline and with published ResNet results (He et al., 2016)
bench = pd.DataFrame([
    ("Baseline CNN (TensorFlow tutorial)",         RESULTS["baseline_test_accuracy"], RESULTS["baseline_params"] / 1e6),
    ("Improved CNN (this project)",                RESULTS["test_accuracy"], total_params / 1e6),
    ("ResNet-20 (He et al., 2016)",                 91.25, 0.27),
    ("ResNet-56 (He et al., 2016)",                 93.03, 0.85),
    ("ResNet-110 (He et al., 2016)",                93.57, 1.7),
], columns=["Model", "Test accuracy (%)", "Params (M)"])
display(bench)

plt.figure(figsize=(8, 3.6))
colors = ["#e45756" if "Improved" in m else ("#f2a541" if "Baseline" in m else "#9db4d0") for m in bench["Model"]]
b = plt.barh(bench["Model"], bench["Test accuracy (%)"], color=colors)
for bar, p in zip(b, bench["Params (M)"]):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}%  ({p:.2f} M params)", va="center", fontsize=8)
plt.xlim(50, 105); plt.xlabel("CIFAR-10 test accuracy (%)"); plt.gca().invert_yaxis()
plt.title("Accuracy vs. benchmarks — accuracy/size trade-off")
plt.savefig(FIG_DIR / "fig07_benchmark_comparison.png"); plt.show()

In [ ]:
# Baseline CNN vs improved CNN: accuracy, size and overfitting (gap between training and validation accuracy)
cmp = pd.DataFrame([
    {"Model": "Baseline CNN", "Test accuracy (%)": RESULTS["baseline_test_accuracy"],
     "Params": RESULTS["baseline_params"], "Train-val gap (pts)": round(RESULTS["baseline_final_train_acc"] - RESULTS["baseline_final_val_acc"], 1)},
    {"Model": "Improved CNN (this project)", "Test accuracy (%)": RESULTS["test_accuracy"],
     "Params": RESULTS["total_params"], "Train-val gap (pts)": round((h["accuracy"][-1] - h["val_accuracy"][-1]) * 100, 1)},
])
RESULTS["improvement_pts"] = round(RESULTS["test_accuracy"] - RESULTS["baseline_test_accuracy"], 2)
display(cmp)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(range(1, len(bh["val_accuracy"]) + 1), np.array(bh["val_accuracy"]) * 100, label="Baseline CNN", color="#f2a541")
a1.plot(range(1, len(h["val_accuracy"]) + 1), np.array(h["val_accuracy"]) * 100, label="Improved CNN", color="#e45756")
a1.xaxis.set_major_locator(plt.MaxNLocator(integer=True)); a1.set_xlabel("epoch"); a1.set_ylabel("validation accuracy (%)"); a1.legend(); a1.grid(alpha=.3); a1.set_title("Validation accuracy per epoch")
bars = a2.bar(cmp["Model"], cmp["Test accuracy (%)"], color=["#f2a541", "#e45756"])
for b in bars: a2.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f"{b.get_height():.1f}%", ha="center")
a2.set_ylim(0, 112); a2.set_ylabel("test accuracy (%)"); a2.set_title(f"Test accuracy: {RESULTS['improvement_pts']:+.1f} points vs Baseline CNN")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig07b_baseline_vs_improved.png"); plt.show()

## 7. Predictions on my own images
Three of my own photos of objects from the CIFAR-10 classes. Before prediction, each photo is cropped to a square around its centre and resized to 32 × 32 pixels, the same size as the training images. The chart next to each photo shows the three most likely classes.

Run the next cell, click **Choose files** and select the three photos.

In [ ]:
def load_rgb(path_or_bytes):
    """Open an image as RGB, turned the right way up (phones store the rotation separately)."""
    img = Image.open(path_or_bytes)
    return ImageOps.exif_transpose(img).convert("RGB")

def preprocess(img, crop_box=None):
    """Image -> the model's input: cropped to a square around the centre and resized to 32 x 32
    (pixel values 0-255). Also returns the small image so it can be shown."""
    if crop_box:                                          # optional box (left, top, right, bottom) cut out first
        img = img.crop(crop_box)
    w, h = img.size; s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    small = img.resize((32, 32), Image.LANCZOS)
    return np.asarray(small, dtype=np.float32), small

def predict_and_plot(paths, title, out_file, crop_boxes=None):
    """Classify each image and plot it: the original, the 32 x 32 model input and the three most likely classes."""
    crop_boxes = crop_boxes or [None] * len(paths)
    n = len(paths); out = []
    fig, axes = plt.subplots(n, 3, figsize=(11, 3.1 * n), gridspec_kw={"width_ratios": [1.3, 1, 1.6]})
    axes = np.atleast_2d(axes)
    for r, (p, cb) in enumerate(zip(paths, crop_boxes)):
        img = load_rgb(p); arr, small = preprocess(img, cb)
        pr = model.predict(arr[None], verbose=0)[0]; top = pr.argsort()[::-1][:3]
        out.append({"image": pathlib.Path(p).name, "prediction": CLASS_NAMES[top[0]],
                    "confidence_%": round(float(pr[top[0]]) * 100, 1),
                    "top3": [(CLASS_NAMES[t], round(float(pr[t]) * 100, 1)) for t in top]})
        axes[r, 0].imshow(img); axes[r, 0].set_title(f"original: {pathlib.Path(p).name}", fontsize=9)
        axes[r, 1].imshow(small, interpolation="nearest"); axes[r, 1].set_title("model input (32×32)", fontsize=9)
        axes[r, 2].barh([CLASS_NAMES[t] for t in top][::-1], pr[top][::-1] * 100,
                        color=["#9db4d0", "#9db4d0", "#2b8a3e"])
        axes[r, 2].set_xlim(0, 100); axes[r, 2].set_xlabel("probability (%)")
        axes[r, 2].set_title(f"→ {CLASS_NAMES[top[0]]} ({pr[top[0]]*100:.1f}%)", fontsize=10, weight="bold")
        for a in axes[r, :2]: a.axis("off")
    plt.suptitle(title, weight="bold"); plt.tight_layout()
    plt.savefig(FIG_DIR / out_file); plt.show()
    return out

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}   # file types accepted as images

def get_uploads(folder):
    """Upload images in Colab and return only the files chosen in this upload, in the order they were
    selected. Images from earlier uploads are deleted so they cannot get mixed in."""
    try:
        from google.colab import files
        up = files.upload()
        if up:
            for old in folder.iterdir():          # delete images from earlier uploads
                if old.is_file(): old.unlink()
            saved = []
            for name, data in up.items():
                p = folder / name; p.write_bytes(data)
                if p.suffix.lower() in IMG_EXT: saved.append(p)
            return saved                          # in the order the files were selected
    except ImportError:                           # not in Colab: use the images already in the folder
        print(f"Not in Colab - reading images from ./{folder}/")
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT)

own_paths = get_uploads(OWN_IMG_DIR)
print("Images:", [p.name for p in own_paths])

In [ ]:
if own_paths:
    RESULTS["own_image_predictions"] = predict_and_plot(own_paths[:3], "Predictions on my own images",
                                                         "fig08_own_image_predictions.png")
    display(pd.DataFrame(RESULTS["own_image_predictions"]))
else:
    print("No images uploaded yet - rerun the previous cell.")

## 8. Predictions on images from the robot's camera (Webots)
Four objects from CIFAR-10 classes are placed at random positions in the Webots arena: a **model airplane**, a **bird** perched on a branch and a **truck**, built from simple shapes in the world file, and a **horse** (the Webots model, coloured brown like most CIFAR-10 horses). An earlier version used a dog, a cat and a white horse; the CNN classified all three as "airplane", which led to the cropped side-view photos. A shiny silver car was also tried: it reflected the blue sky, looked bluish and was classified as "airplane" too, so it was replaced by the bird.

The controller `robot_vision_controller.py` drives the e-puck to a spot beside each object and saves two images in `controllers/robot_vision_controller/images/`:
* `NNN_<object>_view.png`: the full 160 × 120 camera image;
* `NNN_<object>_photo.png`: a square crop centred on the object, worked out from its distance and direction, so the object fills most of the image like a CIFAR-10 photo.

Run the next cell, click **Choose files** and select the four `_photo.png` crops (airplane, bird, truck, horse) together, holding Ctrl to select several files. `_view.png` images can be added as well, to compare the full camera view with the crop. Like my own photos, they are resized to 32 × 32 pixels, the CIFAR-10 size.

Cropping first and then classifying is the detect-then-classify idea proposed for object detection in Webots: the robot finds where the object is, and the CNN says what it is. To try a full `_view.png` image instead, set `WEBOTS_CROPS` to a box `(left, top, right, bottom)` around the object.

In [ ]:
# Choose the animal photos saved by the Webots controller
webots_paths = get_uploads(WEBOTS_IMG_DIR)
print("Images:", [p.name for p in webots_paths])

In [ ]:
WEBOTS_CROPS = None   # only for full _view.png images: one box per image, e.g. [(40, 0, 120, 80)] for 160 x 120
if webots_paths:
    RESULTS["webots_predictions"] = predict_and_plot(webots_paths[:6], "Predictions on e-puck camera images (Webots)",
                                                      "fig09_webots_prediction.png", WEBOTS_CROPS)
    display(pd.DataFrame(RESULTS["webots_predictions"]))
else:
    print("No Webots image uploaded yet - rerun the previous cell.")

**Reading the results.** The Webots photos differ from CIFAR-10 in three ways: the objects are smooth 3D models without the fine detail of real photos (fur, feathers, reflections), about half of each photo is plain sky or floor (in CIFAR-10, a single object in front of blue sky is typical of the airplane class), and at 32 × 32 pixels the object is only a small shape. Objects whose real photos also show smooth painted surfaces and simple shapes (the airplane, the truck) are the easiest for the CNN; animals and birds, whose real photos are full of fur and feather detail, are harder, and colour matters: the brown horse is recognised, while the white horse and the sky-reflecting car were not. Wrong predictions that remain come from **domain shift**: the CNN learned from real photos and has never seen simulated ones. The fix proposed for future work is to fine-tune the CNN on images collected by the robot in Webots.

## 9. Optimisation for running on the robot
The e-puck controller runs every 32 ms (`TIME_STEP = 32`). If the CNN takes too long inside that loop, the motor commands are delayed. Here we measure three ways to make the model cheaper to run:

1. **Post-training quantisation** with TensorFlow Lite: float32 → float16 → dynamic-range int8 → full-integer int8. Smaller numbers make a smaller model that is usually faster on a CPU.
2. **Magnitude pruning:** setting the smallest weights to zero, to see how many can be removed before accuracy drops.
3. **Scheduling:** running the CNN only every N control steps instead of every step.

Speed is measured on the CPU, one image at a time, which is closer to a robot than the Colab GPU. The Keras time includes Python overhead on every call; avoiding that overhead is one reason to use a TensorFlow Lite interpreter on the robot.

In [ ]:
EVAL_N = len(x_test) if not QUICK_TEST else 500            # test images used to measure accuracy
x_eval, y_eval = x_test[:EVAL_N].astype(np.float32), y_test[:EVAL_N]
REP = x_train[:300].astype(np.float32)                     # sample images used to set the int8 scaling

# Export the trained model as a SavedModel, the format the TensorFlow Lite converter reads
export_dir = str(MODEL_DIR / "saved_model")
model.export(export_dir, verbose=False)

def convert(mode):
    """Convert the model to TensorFlow Lite: fp32 (no change), fp16 (16-bit weights), dynamic_int8 (8-bit
    weights) or full_int8 (8-bit weights and calculations; input and output are 0-255 integers)."""
    conv = tf.lite.TFLiteConverter.from_saved_model(export_dir)
    if mode == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types = [tf.float16]
    elif mode == "dynamic_int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == "full_int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = lambda: ([REP[i:i+1]] for i in range(len(REP)))
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.uint8; conv.inference_output_type = tf.uint8
    path = MODEL_DIR / f"cifar10_{mode}.tflite"
    path.write_bytes(conv.convert()); return path

def tflite_runner(path):
    """Return a function that runs one image through a .tflite model on one CPU thread, converting the
    input and output to and from 8-bit integers when the model needs it."""
    itp = tf.lite.Interpreter(model_path=str(path), num_threads=1); itp.allocate_tensors()
    i_d, o_d = itp.get_input_details()[0], itp.get_output_details()[0]
    def run(x):                                            # x: (1,32,32,3) float32 0-255
        if i_d["dtype"] in (np.uint8, np.int8):
            s, z = i_d["quantization"]; info = np.iinfo(i_d["dtype"])
            x = np.clip(np.round(x / s + z), info.min, info.max).astype(i_d["dtype"])
        itp.set_tensor(i_d["index"], x); itp.invoke(); y = itp.get_tensor(o_d["index"])
        if o_d["dtype"] in (np.uint8, np.int8):
            s, z = o_d["quantization"]; y = (y.astype(np.float32) - z) * s
        return y
    return run

def latency_ms(fn, n=200):
    """Average time in milliseconds to classify one image, after 10 warm-up runs."""
    x = x_eval[:1]
    for _ in range(10): fn(x)                              # warm-up
    t = time.perf_counter()
    for i in range(n): fn(x_eval[i % len(x_eval)][None])
    return (time.perf_counter() - t) / n * 1000

def accuracy(fn):
    """Accuracy (%) on the evaluation images, classified one at a time."""
    preds = [fn(x_eval[i][None]).argmax() for i in range(len(x_eval))]
    return float(np.mean(np.array(preds) == y_eval)) * 100

rows = []
keras_path = MODEL_DIR / "cifar10_cnn.keras"; model.save(keras_path)
with tf.device("/CPU:0"):                                  # CPU, one image at a time, like on a robot
    keras_fn = lambda x: model(x, training=False).numpy()
    k_lat = latency_ms(keras_fn, 100)
rows.append({"Variant": "Keras float32 (baseline)", "Size (KB)": keras_path.stat().st_size / 1024,
             "Accuracy (%)": RESULTS["test_accuracy"] if EVAL_N == len(x_test) else accuracy(keras_fn),
             "Latency (ms)": k_lat})

for mode, label in [("fp32", "TFLite float32"), ("fp16", "TFLite float16"),
                    ("dynamic_int8", "TFLite dynamic-range int8"), ("full_int8", "TFLite full-integer int8")]:
    p = convert(mode); fn = tflite_runner(p)
    rows.append({"Variant": label, "Size (KB)": p.stat().st_size / 1024,
                 "Accuracy (%)": accuracy(fn), "Latency (ms)": latency_ms(fn)})
    print(f"{label:28s} done")

opt_df = pd.DataFrame(rows)
opt_df["Max FPS (1 core)"] = 1000 / opt_df["Latency (ms)"]   # images per second on one CPU core
opt_df = opt_df.round(2)
RESULTS["optimisation_table"] = opt_df.to_dict("records")
display(opt_df)

In [ ]:
# Bar charts: file size, time per image and accuracy of each version
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
short = ["Keras fp32", "TFLite fp32", "TFLite fp16", "TFLite dyn-int8", "TFLite int8"]
for ax, col, c in zip(axes, ["Size (KB)", "Latency (ms)", "Accuracy (%)"], ["#4c78a8", "#f58518", "#54a24b"]):
    ax.bar(short, opt_df[col], color=c); ax.set_title(col); ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(opt_df[col]): ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
lo = opt_df["Accuracy (%)"].min(); axes[2].set_ylim(max(0, lo - 10), 100)
plt.suptitle("Quantisation: size / CPU latency / accuracy trade-off (batch 1, single thread)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig10_optimisation.png"); plt.show()

In [ ]:
# Magnitude pruning: set the smallest weights of every Conv/Dense layer to zero (no retraining afterwards)
import gzip
def prune_copy(sparsity):
    """Copy of the model with the given share (0-1) of the smallest weights in each Conv/Dense layer set to 0."""
    m = keras.models.clone_model(model); m.set_weights(model.get_weights())
    for l in m.layers:
        if isinstance(l, (layers.Conv2D, layers.Dense)):
            w = l.get_weights(); k = w[0]
            thr = np.percentile(np.abs(k), sparsity * 100); w[0] = np.where(np.abs(k) < thr, 0, k)
            l.set_weights(w)
    return m

def gz_kb(m):
    """Size of the model's weights after gzip compression, in KB (zeros compress well)."""
    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode="wb") as f:
        for w in m.get_weights(): f.write(w.astype(np.float32).tobytes())
    return len(buf.getvalue()) / 1024

prune_rows = []
for s in [0.0, 0.3, 0.5, 0.7, 0.8, 0.9]:
    m = prune_copy(s); m.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    acc = m.evaluate(x_eval, y_eval, batch_size=256, verbose=0)[1] * 100
    prune_rows.append({"Sparsity (%)": int(s * 100), "Accuracy (%)": round(acc, 2), "Compressed size (KB)": round(gz_kb(m), 1)})
prune_df = pd.DataFrame(prune_rows); RESULTS["pruning_table"] = prune_df.to_dict("records")
display(prune_df)

fig, a1 = plt.subplots(figsize=(7, 3.8)); a2 = a1.twinx()
a1.plot(prune_df["Sparsity (%)"], prune_df["Accuracy (%)"], "o-", color="#54a24b", label="accuracy")
a2.plot(prune_df["Sparsity (%)"], prune_df["Compressed size (KB)"], "s--", color="#4c78a8", label="gzip size")
a1.set_xlabel("weights pruned (%)"); a1.set_ylabel("accuracy (%)", color="#54a24b"); a2.set_ylabel("compressed size (KB)", color="#4c78a8")
plt.title("One-shot magnitude pruning (no fine-tuning)"); fig.tight_layout()
plt.savefig(FIG_DIR / "fig11_pruning.png"); plt.show()
print("Note: pruning with retraining (gradual pruning with tfmot) usually wins back most of the lost accuracy - future work.")

In [ ]:
# Scheduling: running the CNN only every N control steps spreads its cost over several 32 ms Webots time steps
TIME_STEP_MS = 32
best_lat = float(opt_df["Latency (ms)"].min())
base_lat = float(opt_df.loc[0, "Latency (ms)"])
sched = []
for N in [1, 2, 5, 10, 20]:
    for label, lat in [("Keras fp32", base_lat), ("Best TFLite", best_lat)]:
        sched.append({"Run CNN every N steps": N, "Model": label, "Avg ms per step": lat / N,
                      "% of 32 ms step": lat / N / TIME_STEP_MS * 100})
sched_df = pd.DataFrame(sched).round(2); RESULTS["scheduling_table"] = sched_df.to_dict("records")

plt.figure(figsize=(7, 3.8))
for label, g in sched_df.groupby("Model"):
    plt.plot(g["Run CNN every N steps"], g["% of 32 ms step"], "o-", label=label)
plt.axhline(100, color="#e45756", ls="--", lw=1, label="whole time step consumed")
plt.xscale("log"); plt.xticks([1, 2, 5, 10, 20], [1, 2, 5, 10, 20])
plt.xlabel("run CNN every N control steps"); plt.ylabel("% of 32 ms control step")
plt.title("Inference scheduling keeps the control loop responsive"); plt.legend()
plt.savefig(FIG_DIR / "fig12_inference_budget.png"); plt.show()
display(sched_df)

## 10. Summary of results
The key numbers from this run, written as sentences that can be used in the slides and the report.

In [ ]:
# The key results as sentences
lines = []
def pct(key):                                   # a stored accuracy as text, e.g. "84.12%"
    return f"{RESULTS[key]:.2f}%" if key in RESULTS else "not available"

lines.append(f"Baseline CNN: test accuracy {pct('baseline_test_accuracy')}, "
             f"{RESULTS.get('baseline_params', 0):,} parameters.")
for r in RESULTS.get("optimiser_comparison", []):
    lines.append(f"Baseline CNN with {r['Optimiser']}: test accuracy {r['Test acc (%)']:.2f}%, best validation "
                 f"accuracy {r['Best val acc (%)']:.2f}% at epoch {r['Best epoch']}.")
lines.append(f"Improved CNN: test accuracy {pct('test_accuracy')}, {RESULTS.get('total_params', 0):,} parameters, "
             f"{RESULTS.get('improvement_pts', 0):+.2f} percentage points compared with the baseline.")
if "per_class_accuracy" in RESULTS:
    pc = sorted(RESULTS["per_class_accuracy"].items(), key=lambda kv: kv[1])
    lines.append("Hardest classes: " + ", ".join(f"{c} ({a:.1f}%)" for c, a in pc[:2]) +
                 "; easiest classes: " + ", ".join(f"{c} ({a:.1f}%)" for c, a in pc[-2:][::-1]) + ".")
if RESULTS.get("top_confusions"):
    lines.append("Most common mistakes (true -> predicted): " + "; ".join(RESULTS["top_confusions"][:3]) + ".")
for key, label in (("own_image_predictions", "My own images"), ("webots_predictions", "Webots camera image")):
    for p in RESULTS.get(key, []):
        lines.append(f"{label}: {p['image']} -> {p['prediction']} ({p['confidence_%']}%).")
if "optimisation_table" in RESULTS:
    rows = RESULTS["optimisation_table"]
    base = rows[0]
    best = min(rows[1:], key=lambda r: r["Latency (ms)"])
    lines.append(f"Optimisation: {best['Variant']} is {base['Size (KB)'] / best['Size (KB)']:.1f}x smaller "
                 f"({best['Size (KB)']:.0f} KB vs {base['Size (KB)']:.0f} KB) and takes {best['Latency (ms)']:.2f} ms per image "
                 f"vs {base['Latency (ms)']:.2f} ms for Keras on one CPU core, with {best['Accuracy (%)']:.2f}% accuracy.")
RESULTS["summary"] = lines
print("\n".join(lines))

## 11. Save the outputs
All figures, the trained models (`.keras` and `.tflite`) and `results_summary.json` (all the key numbers) are packed into `assignment3_outputs.zip` and downloaded.

In [ ]:
# Save the class names and all key numbers, then zip the figures and models and download them
(MODEL_DIR / "class_names.json").write_text(json.dumps(CLASS_NAMES))
pathlib.Path("results_summary.json").write_text(json.dumps(RESULTS, indent=2, default=str))
print(json.dumps({k: v for k, v in RESULTS.items() if not isinstance(v, (list, dict))}, indent=2))

zip_path = "assignment3_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (FIG_DIR, MODEL_DIR):
        for p in folder.rglob("*"):
            if p.is_file() and "saved_model" not in p.parts: z.write(p)
    z.write("results_summary.json")
print("Packed", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    pass